# Colab starter — Segmentación de heridas en ratas

Correr las celdas en orden. Este notebook clona el repo de GitHub, monta Google Drive (donde están `Images/` y `Masks/`), instala dependencias, y deja todo listo para llamar a los scripts de `src/`.

**Antes de correr:** en Colab, andá a `Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU` para tener GPU disponible.

In [ ]:
# 1) Clonar el repo (reemplazar por tu URL real una vez que lo subas a GitHub)
REPO_URL = "https://github.com/camichahwan/wound_segmentation.git"
!git clone $REPO_URL
%cd wound_segmentation

In [ ]:
# 2) Instalar dependencias
!pip install -q -r requirements.txt

In [ ]:
# 3) Montar Google Drive (te va a pedir autorización la primera vez)
import os, sys

REPO_DIR = "/content/wound_segmentation"
os.chdir(REPO_DIR)  # por si esta celda se corre sola, sin haber corrido la 1 justo antes
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

from config import mount_drive_if_colab, get_data_root, get_images_dir, get_masks_dir, get_checkpoints_dir

mount_drive_if_colab()
print("Data root:", get_data_root())
print("Images:", get_images_dir())
print("Masks:", get_masks_dir())
print("Checkpoints (train.py guarda ACÁ, en Drive -- no en /content, que se borra si se desconecta la sesión):", get_checkpoints_dir())

In [ ]:
# 4) Verificar que el emparejamiento imagen-máscara funciona antes de entrenar nada
from dataset import list_paired_and_unpaired, train_val_test_split

paired, unpaired = list_paired_and_unpaired()
print(f"Imágenes con máscara: {len(paired)}")
print(f"Imágenes sin máscara todavía: {len(unpaired)}")

train_samples, val_samples, test_samples = train_val_test_split(paired)
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

In [ ]:
# 5) Entrenar la U-Net preentrenada (rápido, buen primer experimento)
!cd src && python train.py --model pretrained --encoder resnet34 --epochs 40 --batch_size 8 --image_size 512

In [ ]:
# 6) Entrenar la U-Net desde cero (más lenta en converger, dejarla correr más épocas)
!cd src && python train.py --model scratch --epochs 80 --batch_size 8 --image_size 512

In [ ]:
# 7) Comparar ambos modelos en el set de test
# Los checkpoints ahora se guardan en Drive (ver celda 3, "Checkpoints"), no en /content,
# así que evaluate.py los busca en esa misma carpeta de Drive.
from config import get_checkpoints_dir

ckpt_dir = get_checkpoints_dir()
pretrained_ckpt = os.path.join(ckpt_dir, "unet_pretrained.pt")
scratch_ckpt = os.path.join(ckpt_dir, "unet_scratch.pt")

!cd src && python evaluate.py --checkpoints "{pretrained_ckpt}" "{scratch_ckpt}"

In [ ]:
# 8) Post-procesamiento + ensemble: ¿ayudan a los errores que NO son de heridas chicas?
# Reutiliza los dos checkpoints ya entrenados (no reentrena nada). Compara contra
# unet_pretrained solo con test pareado de Wilcoxon -- ver outputs/error_analysis/
# despues de correrlo para el reporte con la decision.
!cd src && python postprocess_and_ensemble.py

# Enfoque en dos etapas (heridas chicas/tempranas)

Ver `outputs/error_analysis/heridas_chicas_resize.md`: las heridas más chicas del dataset quedan en ~6x6 píxeles después del resize a 512x512 que usa el pipeline de una sola etapa -- por debajo de lo que la red puede ver. Estas celdas entrenan y evalúan un enfoque en dos etapas: una localización gruesa a mayor resolución, seguida de un recorte + segmentación fina sobre esa región. Correr en orden, cada celda depende de la anterior.

In [ ]:
# 9) Etapa 1: entrenar el localizador a mayor resolución (1024, no 512)
# batch_size mas chico que en la celda 5 porque 1024x1024 pesa mucho mas en memoria de GPU.
# Si tira error de memoria (OOM) en la T4, bajar --batch_size a 2.
!cd src && python train.py --model pretrained --encoder resnet34 --epochs 30 --batch_size 4 \
    --image_size 1024 --run_name unet_localizer_1024

In [ ]:
# 10) Generar el dataset de recortes (a partir de las máscaras reales) para entrenar la etapa 2
!cd src && python make_crop_dataset.py --crop_size 512

In [ ]:
# 11) Etapa 2: fine-tuning de unet_pretrained sobre los recortes (parte de los pesos ya entrenados,
# no de cero -- ver --init_checkpoint). WOUND_DATA_ROOT apunta train.py al dataset de recortes
# generado en la celda anterior, en vez de a las imágenes completas.
import os
from config import get_crops_data_root, get_checkpoints_dir, get_data_root

os.environ["WOUND_DATA_ROOT"] = get_crops_data_root()
print("Entrenando etapa 2 sobre:", os.environ["WOUND_DATA_ROOT"])

ckpt_dir = get_checkpoints_dir()
pretrained_ckpt = os.path.join(ckpt_dir, "unet_pretrained.pt")

!cd src && python train.py --model pretrained --encoder resnet34 --epochs 30 --batch_size 8 \
    --image_size 512 --run_name unet_stage2_finetuned --init_checkpoint "{pretrained_ckpt}"

# IMPORTANTE: sacar el override apenas termina, para que las celdas siguientes (evaluación de
# punta a punta) vuelvan a usar las imágenes completas originales, no los recortes.
del os.environ["WOUND_DATA_ROOT"]
print("WOUND_DATA_ROOT reseteado. Data root de nuevo:", get_data_root())

In [ ]:
# 12) Evaluación de punta a punta: dos etapas vs. unet_pretrained solo, sobre las imágenes
# de test ORIGINALES completas (no recortes) -- esta es la comparación real.
ckpt_dir = get_checkpoints_dir()
stage1_ckpt = os.path.join(ckpt_dir, "unet_localizer_1024.pt")
stage2_ckpt = os.path.join(ckpt_dir, "unet_stage2_finetuned.pt")

!cd src && python evaluate_two_stage.py --stage1_checkpoint "{stage1_ckpt}" --stage2_checkpoint "{stage2_ckpt}"